## Connecting to SQLite Search Database

In [1]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [2]:
sqlite_index.count()

79

In [3]:
results = sqlite_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'I missed the first homework - can I still get a certificate?',
 'Do we submit 2 projects, what does attempt 1 and 2 mean?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'I am using Azure OpenAI and I am still getting an error of Error code: 400 - {\'error\': {\'message\': "Missing required parameter: \'tools[0].function\'.", \'type\': \'invalid_request_error\', \'param\': \'tools[0].function\', \'code\': \'missing_required_parameter\'}}?']

In [6]:
import os
from rag_helper import LMStudioRAG # Since using local AI
from openai import OpenAI

from dotenv import load_dotenv
load_dotenv()

openai_client = OpenAI(
    api_key=os.getenv("LMSTUDIO_API_KEY"),
    base_url=os.getenv("LMSTUDIO_HOST")
)

model = "qwen/qwen3.5-9b"

assistant = LMStudioRAG(
    index=sqlite_index,
    llm_client=openai_client,
    model=model
)

Note: This code skips both the fit call and the data loading. The index is already populated by the ingestion notebook, so we just connect to the database file.

In [7]:
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

Based on the provided context, yes, you can still join the course after it started (or even if you discover it late), but there are conditions regarding certificates:

*   **If you want a certificate:** You must submit your project while the course is still accepting submissions. Additionally, you cannot get a certificate via self-paced mode; you must finish with a "live" cohort to receive one because peer reviews for capstone projects only occur while the course is running.
*   **Without a certificate concern:** The context notes that for the LLM Zoomcamp specifically, registration is just to gauge interest before the start date and is not checked against a list, implying you can start learning immediately without formal registration, but it does not explicitly detail joining rules for after the official start date outside of the certificate requirements mentioned above.


The answer should be similar to what we got with minsearch. But now the data comes from a persistent index - no fetching, no processing, no indexing at startup. And we didn't have to rewrite any of the RAG logic - just swapped the index.

The modular design splits the work cleanly:

    ingest.py handles data loading and indexing
    rag_helper.py handles the RAG pipeline
    the notebooks wire them together

This works because sqlitesearch follows the same API as minsearch - both have a search method that takes a query, boost_dict, filter_dict, and num_results. If the API were different, we'd need to subclass RAGBase and override the search method to adapt to the new backend.

In [8]:
# Cleaning up
sqlite_index.close()